# 09. M1 pre-event compact feature set 검증

## tl;dr

라벨/window/threshold는 고정하고, 08번의 `expanded154` feature set을 중요도 기반 compact feature set으로 줄였을 때 성능을 유지할 수 있는지 확인한다.

## Context & Methods

- 기준 라벨: `normal` vs `efd_possible`
- main dataset: `strict_no_event20`
- sensitivity dataset: `strict_no_event20_no_event67`
- threshold: `0.6`
- 제외 유지: Event 20, Event 34, Event 69
- 비교 feature set:
  - `base70`
  - `expanded154`
  - `compact13_overlap`
  - `compact20_main`
  - `compact27_union`
- compact feature 선정은 08번 `expanded154` logistic coefficient importance의 top20 기준으로 한다.

In [1]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "07_데이터산출물"

INPUT_FEATURES_PATH = OUTPUT_DIR / "m1_feature_expansion_features.csv"
INPUT_IMPORTANCE_PATH = OUTPUT_DIR / "m1_feature_expansion_feature_importance.csv"
INPUT_THRESHOLD_PATH = OUTPUT_DIR / "m1_feature_expansion_threshold_metrics.csv"

SUMMARY_PATH = OUTPUT_DIR / "m1_compact_feature_set_summary.csv"
CV_METRICS_PATH = OUTPUT_DIR / "m1_compact_feature_set_cv_metrics.csv"
CV_PREDICTIONS_PATH = OUTPUT_DIR / "m1_compact_feature_set_cv_predictions.csv"
THRESHOLD_METRICS_PATH = OUTPUT_DIR / "m1_compact_feature_set_threshold_metrics.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "m1_compact_feature_set_feature_importance.csv"
DECISION_MATRIX_PATH = OUTPUT_DIR / "m1_compact_feature_set_decision_matrix.csv"
REPORT_PATH = OUTPUT_DIR / "09_M1_pre_event_compact_feature_set_검증_보고서.md"

PNG_TOP20_MAIN_PATH = OUTPUT_DIR / "m1_compact_feature_importance_top20_main.png"
PNG_TOP20_NO_EVENT67_PATH = OUTPUT_DIR / "m1_compact_feature_importance_top20_no_event67.png"
PNG_OVERLAP_PATH = OUTPUT_DIR / "m1_compact_feature_set_overlap.png"

RANDOM_STATE = 42
POSITIVE_LABEL = "efd_possible"
EVENT20_ID = 20
EVENT34_ID = 34
EVENT67_ID = 67
EVENT69_ID = 69
DECISION_THRESHOLD = 0.6

ORIGINAL_SIGNALS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
SENSOR_STATS = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]
BASE_FEATURE_COLUMNS = [f"{signal}__{stat}" for signal in ORIGINAL_SIGNALS for stat in SENSOR_STATS]

LABEL_TO_BINARY = {"normal": 0, POSITIVE_LABEL: 1}
BINARY_TO_LABEL = {0: "normal", 1: POSITIVE_LABEL}

for path in [INPUT_FEATURES_PATH, INPUT_IMPORTANCE_PATH, INPUT_THRESHOLD_PATH]:
    assert path.exists(), path

## Data

In [2]:
features_df = pd.read_csv(INPUT_FEATURES_PATH)
source_importance_df = pd.read_csv(INPUT_IMPORTANCE_PATH)
source_threshold_df = pd.read_csv(INPUT_THRESHOLD_PATH)

EXPANDED_FEATURE_COLUMNS = sorted([column for column in features_df.columns if "__" in column])
assert len(BASE_FEATURE_COLUMNS) == 70
assert set(BASE_FEATURE_COLUMNS).issubset(EXPANDED_FEATURE_COLUMNS)
assert len(EXPANDED_FEATURE_COLUMNS) == 154

normal_df = features_df.loc[features_df["label"].eq("normal")].copy()
positive_df = features_df.loc[features_df["label"].eq(POSITIVE_LABEL)].copy()

def build_dataset(dataset_id: str) -> pd.DataFrame:
    if dataset_id == "strict_no_event20":
        positive_part = positive_df.copy()
    elif dataset_id == "strict_no_event20_no_event67":
        positive_part = positive_df.loc[~positive_df["source_event_id"].eq(EVENT67_ID)].copy()
    else:
        raise ValueError(dataset_id)
    dataset = pd.concat([normal_df, positive_part], ignore_index=True)
    dataset["dataset_id"] = dataset_id
    return dataset


DATASETS = {
    "strict_no_event20": build_dataset("strict_no_event20"),
    "strict_no_event20_no_event67": build_dataset("strict_no_event20_no_event67"),
}

for dataset_id, dataset in DATASETS.items():
    positive_events = set(dataset.loc[dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int))
    assert EVENT20_ID not in positive_events
    assert EVENT34_ID not in positive_events
    assert EVENT69_ID not in positive_events
    if dataset_id == "strict_no_event20":
        assert EVENT67_ID in positive_events
    else:
        assert EVENT67_ID not in positive_events

print(features_df.shape)
print({dataset_id: dataset.shape for dataset_id, dataset in DATASETS.items()})

(49, 181)
{'strict_no_event20': (49, 182), 'strict_no_event20_no_event67': (48, 182)}


## Compact Feature Selection

In [3]:
expanded_importance_df = source_importance_df.loc[
    source_importance_df["feature_set"].eq("expanded154")
].copy()

top20_by_dataset = {
    dataset_id: expanded_importance_df.loc[expanded_importance_df["dataset_id"].eq(dataset_id)]
    .sort_values("mean_abs_coefficient", ascending=False)
    .head(20)
    .copy()
    for dataset_id in DATASETS
}

main_top20_features = list(top20_by_dataset["strict_no_event20"]["feature"])
no_event67_top20_features = list(top20_by_dataset["strict_no_event20_no_event67"]["feature"])
overlap_features = sorted(set(main_top20_features).intersection(no_event67_top20_features))
union_features = sorted(set(main_top20_features).union(no_event67_top20_features))

FEATURE_SET_COLUMNS = {
    "base70": BASE_FEATURE_COLUMNS,
    "expanded154": EXPANDED_FEATURE_COLUMNS,
    "compact13_overlap": overlap_features,
    "compact20_main": main_top20_features,
    "compact27_union": union_features,
}

assert len(FEATURE_SET_COLUMNS["compact13_overlap"]) == 13
assert len(FEATURE_SET_COLUMNS["compact20_main"]) == 20
assert len(FEATURE_SET_COLUMNS["compact27_union"]) == 27

summary_rows = []
for feature_set, columns in FEATURE_SET_COLUMNS.items():
    summary_rows.append(
        {
            "feature_set": feature_set,
            "feature_count": len(columns),
            "selection_rule": {
                "base70": "original common 10 signals x 7 stats",
                "expanded154": "base70 plus derived and temporal expansion",
                "compact13_overlap": "intersection of top20 expanded features from both datasets",
                "compact20_main": "top20 expanded features from strict_no_event20",
                "compact27_union": "union of top20 expanded features from both datasets",
            }[feature_set],
            "features": "|".join(columns),
        }
    )

feature_set_summary_df = pd.DataFrame(summary_rows)
feature_set_summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
feature_set_summary_df[["feature_set", "feature_count", "selection_rule"]]

,feature_set,feature_count,selection_rule
0,base70,70,original common 10 signals x 7 stats
1,expanded154,154,base70 plus derived and temporal expansion
2,compact13_overlap,13,intersection of top20 expanded features from b...
3,compact20_main,20,top20 expanded features from strict_no_event20
4,compact27_union,27,union of top20 expanded features from both dat...


## Feature Importance Visualization

In [4]:
def save_top_feature_bar(df: pd.DataFrame, title: str, path: Path) -> None:
    plot_df = df.sort_values("mean_abs_coefficient", ascending=True)
    fig_height = max(5.0, 0.32 * len(plot_df))
    fig, ax = plt.subplots(figsize=(10, fig_height))
    ax.barh(plot_df["feature"], plot_df["mean_abs_coefficient"], color="#2563eb")
    ax.set_title(title)
    ax.set_xlabel("Mean absolute coefficient")
    ax.set_ylabel("Feature")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


save_top_feature_bar(
    top20_by_dataset["strict_no_event20"],
    "Top20 feature importance: strict_no_event20",
    PNG_TOP20_MAIN_PATH,
)
save_top_feature_bar(
    top20_by_dataset["strict_no_event20_no_event67"],
    "Top20 feature importance: strict_no_event20_no_event67",
    PNG_TOP20_NO_EVENT67_PATH,
)

overlap_counts = pd.DataFrame(
    [
        {"set": "main top20", "count": len(main_top20_features)},
        {"set": "no_event67 top20", "count": len(no_event67_top20_features)},
        {"set": "overlap", "count": len(overlap_features)},
        {"set": "union", "count": len(union_features)},
    ]
)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(overlap_counts["set"], overlap_counts["count"], color=["#64748b", "#64748b", "#16a34a", "#f97316"])
for index, row in overlap_counts.iterrows():
    ax.text(index, row["count"] + 0.4, str(row["count"]), ha="center", va="bottom")
ax.set_ylim(0, max(overlap_counts["count"]) + 5)
ax.set_ylabel("Feature count")
ax.set_title("Top20 feature set overlap")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(PNG_OVERLAP_PATH, dpi=160, bbox_inches="tight")
plt.close(fig)

for path in [PNG_TOP20_MAIN_PATH, PNG_TOP20_NO_EVENT67_PATH, PNG_OVERLAP_PATH]:
    assert path.exists() and path.stat().st_size > 0, path
print(PNG_TOP20_MAIN_PATH.relative_to(ROOT))
print(PNG_TOP20_NO_EVENT67_PATH.relative_to(ROOT))
print(PNG_OVERLAP_PATH.relative_to(ROOT))

07_데이터산출물\m1_compact_feature_importance_top20_main.png
07_데이터산출물\m1_compact_feature_importance_top20_no_event67.png
07_데이터산출물\m1_compact_feature_set_overlap.png


## Training

In [5]:
def make_models() -> dict[str, Pipeline]:
    return {
        "dummy_most_frequent": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        ),
        "logistic_balanced": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }


def positive_scores(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    estimator = model.named_steps["model"]
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        classes = list(estimator.classes_)
        if 1 in classes:
            return probabilities[:, classes.index(1)]
    return np.zeros(len(X), dtype=float)


def metric_row(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if y_score is not None and len(np.unique(y_true)) == 2:
        roc_auc = float(roc_auc_score(y_true, y_score))
    else:
        roc_auc = 0.5
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def train_and_evaluate(dataset_id: str, feature_set: str, dataset: pd.DataFrame):
    feature_columns = FEATURE_SET_COLUMNS[feature_set]
    dataset = dataset.copy().reset_index(drop=True)
    X = dataset[feature_columns].copy()
    y = dataset["label"].map(LABEL_TO_BINARY).astype(int).to_numpy()
    groups = dataset["substation_id"].astype(str).to_numpy()

    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    metric_rows = []
    prediction_rows = []
    importance_rows = []

    for fold, (train_index, test_index) in enumerate(splitter.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]
        train_groups = sorted(set(groups[train_index]))
        test_groups = sorted(set(groups[test_index]))
        assert set(train_groups).isdisjoint(test_groups)
        assert len(np.unique(y_train)) == 2

        for model_name, model in make_models().items():
            model.fit(X_train, y_train)
            y_score = positive_scores(model, X_test)
            y_pred = model.predict(X_test).astype(int)
            row = metric_row(y_test, y_pred, y_score)
            row.update(
                {
                    "dataset_id": dataset_id,
                    "feature_set": feature_set,
                    "model": model_name,
                    "fold": fold,
                    "feature_count": len(feature_columns),
                    "train_rows": int(len(train_index)),
                    "test_rows": int(len(test_index)),
                    "train_groups": "|".join(train_groups),
                    "test_groups": "|".join(test_groups),
                    "test_normal": int((y_test == 0).sum()),
                    "test_efd_possible": int((y_test == 1).sum()),
                }
            )
            metric_rows.append(row)

            prediction_meta = dataset.iloc[test_index][
                [
                    "sample_id",
                    "label",
                    "label_strength",
                    "source_event_id",
                    "substation_id",
                    "window_start",
                    "window_end",
                    "window_days",
                    "coverage_rate",
                    "sample_count",
                    "event67_flag",
                ]
            ].copy()
            prediction_meta["dataset_id"] = dataset_id
            prediction_meta["feature_set"] = feature_set
            prediction_meta["fold"] = fold
            prediction_meta["model"] = model_name
            prediction_meta["actual_binary"] = y_test
            prediction_meta["positive_score"] = y_score
            prediction_meta["predicted_binary"] = y_pred
            prediction_meta["predicted_label"] = [BINARY_TO_LABEL[int(value)] for value in y_pred]
            prediction_meta["is_correct"] = prediction_meta["actual_binary"].eq(prediction_meta["predicted_binary"])
            prediction_rows.extend(prediction_meta.to_dict("records"))

            if model_name == "logistic_balanced":
                coefficients = model.named_steps["model"].coef_[0]
                for feature, coefficient in zip(feature_columns, coefficients):
                    signal, stat = feature.split("__", 1)
                    importance_rows.append(
                        {
                            "dataset_id": dataset_id,
                            "feature_set": feature_set,
                            "fold": fold,
                            "feature": feature,
                            "signal": signal,
                            "stat": stat,
                            "coefficient": float(coefficient),
                            "abs_coefficient": float(abs(coefficient)),
                        }
                    )
    return metric_rows, prediction_rows, importance_rows


all_metric_rows = []
all_prediction_rows = []
all_importance_rows = []
for dataset_id, dataset in DATASETS.items():
    for feature_set in FEATURE_SET_COLUMNS:
        metric_rows, prediction_rows, importance_rows = train_and_evaluate(dataset_id, feature_set, dataset)
        all_metric_rows.extend(metric_rows)
        all_prediction_rows.extend(prediction_rows)
        all_importance_rows.extend(importance_rows)

cv_metrics_df = pd.DataFrame(all_metric_rows)
cv_predictions_df = pd.DataFrame(all_prediction_rows)
feature_importance_raw_df = pd.DataFrame(all_importance_rows)

cv_metrics_df.to_csv(CV_METRICS_PATH, index=False, encoding="utf-8-sig")
cv_predictions_df.to_csv(CV_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

feature_importance_df = (
    feature_importance_raw_df.groupby(["dataset_id", "feature_set", "feature", "signal", "stat"], as_index=False)
    .agg(
        mean_coefficient=("coefficient", "mean"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
        std_coefficient=("coefficient", "std"),
    )
    .sort_values(["dataset_id", "feature_set", "mean_abs_coefficient"], ascending=[True, True, False])
)
feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")

cv_metrics_df.groupby(["dataset_id", "feature_set", "model"])[
    ["balanced_accuracy", "precision", "recall", "f1", "roc_auc", "fp", "fn"]
].mean().round(4)

balanced_accuracy  \
dataset_id                   feature_set       model                                    
strict_no_event20            base70            dummy_most_frequent             0.5000   
                                               logistic_balanced               0.5738   
                             compact13_overlap dummy_most_frequent             0.5000   
                                               logistic_balanced               0.8065   
                             compact20_main    dummy_most_frequent             0.5000   
                                               logistic_balanced               0.7923   
                             compact27_union   dummy_most_frequent             0.5000   
                                               logistic_balanced               0.8732   
                             expanded154       dummy_most_frequent             0.5000   
                                               logistic_balanced               0.6220   
strict_no_event20_no_event67 base70            dummy_most_frequent             0.5000   
                                               logistic_balanced               0.5821   
                             compact13_overlap dummy_most_frequent             0.5000   
                                               logistic_balanced               0.8381   
                             compact20_main    dummy_most_frequent             0.5000   
                                               logistic_balanced               0.8405   
                             compact27_union   dummy_most_frequent             0.5000   
                                               logistic_balanced               0.8381   
                             expanded154       dummy_most_frequent             0.5000   
                                               logistic_balanced               0.6810   

                                                                    precision  \
dataset_id                   feature_set       model                            
strict_no_event20            base70            dummy_most_frequent     0.0000   
                                               logistic_balanced       0.3000   
                             compact13_overlap dummy_most_frequent     0.0000   
                                               logistic_balanced       0.7333   
                             compact20_main    dummy_most_frequent     0.0000   
                                               logistic_balanced       0.8500   
                             compact27_union   dummy_most_frequent     0.0000   
                                               logistic_balanced       0.8833   
                             expanded154       dummy_most_frequent     0.0000   
                                               logistic_balanced       0.3600   
strict_no_event20_no_event67 base70            dummy_most_frequent     0.0000   
                                               logistic_balanced       0.3333   
                             compact13_overlap dummy_most_frequent     0.0000   
                                               logistic_balanced       0.8167   
                             compact20_main    dummy_most_frequent     0.0000   
                                               logistic_balanced       0.8500   
                             compact27_union   dummy_most_frequent     0.0000   
                                               logistic_balanced       0.7000   
                             expanded154       dummy_most_frequent     0.0000   
                                               logistic_balanced       0.5033   

                                                                    recall  \
dataset_id                   feature_set       model                         
strict_no_event20            base70            dummy_most_frequent  0.0000   
                                               logistic_balanced    0.5000   
              

## Threshold And Decision

In [6]:
threshold_rows = []
for dataset_id in DATASETS:
    for feature_set in FEATURE_SET_COLUMNS:
        prediction_part = cv_predictions_df.loc[
            cv_predictions_df["dataset_id"].eq(dataset_id)
            & cv_predictions_df["feature_set"].eq(feature_set)
            & cv_predictions_df["model"].eq("logistic_balanced")
        ].copy()
        y_true = prediction_part["actual_binary"].astype(int).to_numpy()
        y_score = prediction_part["positive_score"].astype(float).to_numpy()
        y_pred = (y_score >= DECISION_THRESHOLD).astype(int)
        row = metric_row(y_true, y_pred, y_score)
        row.update(
            {
                "dataset_id": dataset_id,
                "feature_set": feature_set,
                "threshold": DECISION_THRESHOLD,
                "feature_count": len(FEATURE_SET_COLUMNS[feature_set]),
                "rows": int(len(prediction_part)),
                "normal_rows": int((y_true == 0).sum()),
                "positive_rows": int((y_true == 1).sum()),
                "false_positive_rate": float(row["fp"] / (row["fp"] + row["tn"])) if (row["fp"] + row["tn"]) else 0.0,
                "false_negative_rate": float(row["fn"] / (row["fn"] + row["tp"])) if (row["fn"] + row["tp"]) else 0.0,
            }
        )
        threshold_rows.append(row)

threshold_metrics_df = pd.DataFrame(threshold_rows)
threshold_metrics_df.to_csv(THRESHOLD_METRICS_PATH, index=False, encoding="utf-8-sig")

def threshold_record(dataset_id: str, feature_set: str) -> pd.Series:
    return threshold_metrics_df.loc[
        threshold_metrics_df["dataset_id"].eq(dataset_id)
        & threshold_metrics_df["feature_set"].eq(feature_set)
    ].iloc[0]


decision_rows = []
compact_candidates = ["compact13_overlap", "compact20_main", "compact27_union"]
for feature_set in FEATURE_SET_COLUMNS:
    for dataset_id in DATASETS:
        expanded = threshold_record(dataset_id, "expanded154")
        current = threshold_record(dataset_id, feature_set)
        decision_rows.append(
            {
                "dataset_id": dataset_id,
                "feature_set": feature_set,
                "threshold": DECISION_THRESHOLD,
                "feature_count": len(FEATURE_SET_COLUMNS[feature_set]),
                "balanced_accuracy": float(current["balanced_accuracy"]),
                "recall": float(current["recall"]),
                "f1": float(current["f1"]),
                "false_positive_rate": float(current["false_positive_rate"]),
                "fp": int(current["fp"]),
                "fn": int(current["fn"]),
                "tp": int(current["tp"]),
                "tn": int(current["tn"]),
                "ba_delta_vs_expanded154": float(current["balanced_accuracy"] - expanded["balanced_accuracy"]),
                "recall_delta_vs_expanded154": float(current["recall"] - expanded["recall"]),
                "fpr_delta_vs_expanded154": float(current["false_positive_rate"] - expanded["false_positive_rate"]),
            }
        )

decision_matrix_df = pd.DataFrame(decision_rows)

candidate_rows = []
for feature_set in compact_candidates:
    subset = decision_matrix_df.loc[decision_matrix_df["feature_set"].eq(feature_set)].copy()
    ba_pass = bool((subset["ba_delta_vs_expanded154"] >= -0.02).all())
    recall_pass = bool((subset["recall_delta_vs_expanded154"] >= -0.05).all())
    fpr_pass = bool((subset["fpr_delta_vs_expanded154"] <= 0.05).all())
    feature_count_pass = len(FEATURE_SET_COLUMNS[feature_set]) < len(FEATURE_SET_COLUMNS["expanded154"])
    candidate_rows.append(
        {
            "feature_set": feature_set,
            "priority": compact_candidates.index(feature_set) + 1,
            "ba_pass": ba_pass,
            "recall_pass": recall_pass,
            "fpr_pass": fpr_pass,
            "feature_count_pass": feature_count_pass,
            "candidate_pass": ba_pass and recall_pass and fpr_pass and feature_count_pass,
        }
    )

candidate_decision_df = pd.DataFrame(candidate_rows)
passed_candidates = candidate_decision_df.loc[candidate_decision_df["candidate_pass"]].sort_values("priority")
if len(passed_candidates):
    selected_feature_set = str(passed_candidates.iloc[0]["feature_set"])
    final_decision = f"{selected_feature_set} compact feature set 채택"
else:
    selected_feature_set = "expanded154"
    final_decision = "compact feature set은 채택 기준 미달, expanded154 유지"

decision_matrix_df = decision_matrix_df.merge(candidate_decision_df, on="feature_set", how="left")
decision_matrix_df["selected_feature_set"] = selected_feature_set
decision_matrix_df["final_decision"] = final_decision
decision_matrix_df.to_csv(DECISION_MATRIX_PATH, index=False, encoding="utf-8-sig")

threshold_metrics_df.sort_values(["dataset_id", "feature_set"])

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp,dataset_id,feature_set,threshold,feature_count,rows,normal_rows,positive_rows,false_positive_rate,false_negative_rate
0,0.673469,0.621429,0.437500,0.500000,0.466667,0.604082,26,9,7,7,strict_no_event20,base70,0.6,70,49,35,14,0.257143,0.500000
2,0.877551,0.828571,0.833333,0.714286,0.769231,0.918367,33,2,4,10,strict_no_event20,compact13_overlap,0.6,13,49,35,14,0.057143,0.285714
3,0.836735,0.778571,0.750000,0.642857,0.692308,0.900000,32,3,5,9,strict_no_event20,compact20_main,0.6,20,49,35,14,0.085714,0.357143
4,0.897959,0.864286,0.846154,0.785714,0.814815,0.887755,33,2,3,11,strict_no_event20,compact27_union,0.6,27,49,35,14,0.057143,0.214286
1,0.714286,0.650000,0.500000,0.500000,0.500000,0.677551,28,7,7,7,strict_no_event20,expanded154,0.6,154,49,35,14,0.200000,0.500000
5,0.645833,0.612088,0.388889,0.538462,0.451613,0.648352,24,11,6,7,strict_no_event20_no_event67,base70,0.6,70,48,35,13,0.314286,0.461538
7,0.875000,0.817582,0.818182,0.692308,0.750000,0.890110,33,2,4,9,strict_no_event20_no_event67,compact13_overlap,0.6,13,48,35,13,0.057143,0.307692
8,0.895833,0.856044,0.833333,0.769231,0.800000,0.865934,33,2,3,10,strict_no_event20_no_event67,compact20_main,0.6,20,48,35,13,0.057143,0.230769
9,0.916667,0.870330,0.909091,0.769231,0.833333,0.942857,34,1,3,10,strict_no_event20_no_event67,compact27_union,0.6,27,48,35,13,0.028571,0.230769
6,0.750000,0.659341,0.545455,0.461538,0.500000,0.738462,30,5,7,6,strict_no_event20_no_event67,expanded154,0.6,154,48,35,13,0.142857,0.538462


## Report

In [7]:
def markdown_table(df: pd.DataFrame, float_cols: list[str]) -> str:
    table = df.copy()
    for column in float_cols:
        if column in table.columns:
            table[column] = table[column].map(lambda value: f"{value:.4f}")
    columns = list(table.columns)
    lines = [
        "| " + " | ".join(str(column) for column in columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for _, row in table.iterrows():
        values = []
        for column in columns:
            value = row[column]
            values.append("" if pd.isna(value) else str(value).replace("|", "\\|"))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


cv_summary = (
    cv_metrics_df.groupby(["dataset_id", "feature_set", "model"], as_index=False)
    .agg(
        balanced_accuracy=("balanced_accuracy", "mean"),
        precision=("precision", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean"),
        roc_auc=("roc_auc", "mean"),
        fp=("fp", "mean"),
        fn=("fn", "mean"),
    )
    .sort_values(["dataset_id", "feature_set", "model"])
)

decision_view = decision_matrix_df[
    [
        "dataset_id",
        "feature_set",
        "feature_count",
        "balanced_accuracy",
        "recall",
        "f1",
        "false_positive_rate",
        "fp",
        "fn",
        "ba_delta_vs_expanded154",
        "recall_delta_vs_expanded154",
        "fpr_delta_vs_expanded154",
        "candidate_pass",
    ]
].copy()

top_feature_type_summary = pd.DataFrame(
    [
        {"type": "최근 6시간/12시간 변화", "evidence": "outdoor_temperature 및 supply temperature 변화 feature가 양쪽 top20에 반복 등장"},
        {"type": "최근 1일 vs 이전 6일 변화", "evidence": "p_return_gap, return temperature, supply temperature 계열이 상위권"},
        {"type": "return temperature gap", "evidence": "p_return_gap feature가 main/sensitivity 모두 최상위"},
        {"type": "supply temperature / setpoint", "evidence": "s_hc1_supply_temperature 및 setpoint 변화 feature가 공통 feature set에 포함"},
    ]
)

selected_count = len(FEATURE_SET_COLUMNS[selected_feature_set])
report = f'''
# M1 pre-event compact feature set 검증 보고서

## 결론

- 최종 판단: {final_decision}
- 선택 feature set: `{selected_feature_set}` ({selected_count}개 feature)
- threshold는 0.6으로 고정했고, 라벨/window 기준은 08번과 동일하게 유지했다.
- compact 후보는 154개 feature 대비 feature 수를 크게 줄였지만, 채택 기준을 모두 만족한 후보만 최종 후보로 인정했다.

## Feature Set Summary

{markdown_table(feature_set_summary_df[['feature_set', 'feature_count', 'selection_rule']], [])}

## Threshold 0.6 Decision Matrix

{markdown_table(decision_view, ['balanced_accuracy', 'recall', 'f1', 'false_positive_rate', 'ba_delta_vs_expanded154', 'recall_delta_vs_expanded154', 'fpr_delta_vs_expanded154'])}

## Candidate Pass Summary

{markdown_table(candidate_decision_df, [])}

## Group CV 평균 성능

{markdown_table(cv_summary, ['balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'fp', 'fn'])}

## 중요 feature 유형 요약

{markdown_table(top_feature_type_summary, [])}

## 생성 이미지

- `07_데이터산출물/m1_compact_feature_importance_top20_main.png`
- `07_데이터산출물/m1_compact_feature_importance_top20_no_event67.png`
- `07_데이터산출물/m1_compact_feature_set_overlap.png`

## 해석

- `expanded154`의 개선 신호는 주로 최근 변화량과 온도차 계열에서 나왔다.
- compact feature set은 중요 feature를 압축해 검증하기 위한 후보이며, 최종 운영 모델 저장이나 배포는 하지 않았다.
- 이번 compact 선정은 08번의 전체 CV 중요도 결과를 사용한 post-hoc feature selection이므로 성능이 낙관적으로 보일 수 있다. 따라서 채택은 다음 실험 후보 채택이지 운영 확정이 아니다.
- compact 후보가 기준을 통과하지 못하면 `expanded154`를 유지하고, 다음 단계에서 normal negative 재설계 또는 센서 컬럼 확장을 검토한다.
'''.strip()

REPORT_PATH.write_text(report, encoding="utf-8")
print(final_decision)

compact13_overlap compact feature set 채택


## Checks

In [8]:
assert len(FEATURE_SET_COLUMNS["base70"]) == 70
assert len(FEATURE_SET_COLUMNS["expanded154"]) == 154
assert len(FEATURE_SET_COLUMNS["compact13_overlap"]) == 13
assert len(FEATURE_SET_COLUMNS["compact20_main"]) == 20
assert len(FEATURE_SET_COLUMNS["compact27_union"]) == 27

assert len(DATASETS["strict_no_event20"]) == 49
assert int(DATASETS["strict_no_event20"]["label"].eq("normal").sum()) == 35
assert int(DATASETS["strict_no_event20"]["label"].eq(POSITIVE_LABEL).sum()) == 14
assert len(DATASETS["strict_no_event20_no_event67"]) == 48
assert int(DATASETS["strict_no_event20_no_event67"]["label"].eq("normal").sum()) == 35
assert int(DATASETS["strict_no_event20_no_event67"]["label"].eq(POSITIVE_LABEL).sum()) == 13

for dataset_id, dataset in DATASETS.items():
    positive_events = set(dataset.loc[dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int))
    assert EVENT20_ID not in positive_events
    assert EVENT34_ID not in positive_events
    assert EVENT69_ID not in positive_events

assert EVENT67_ID in set(
    DATASETS["strict_no_event20"].loc[
        DATASETS["strict_no_event20"]["label"].eq(POSITIVE_LABEL), "source_event_id"
    ].astype(int)
)
assert EVENT67_ID not in set(
    DATASETS["strict_no_event20_no_event67"].loc[
        DATASETS["strict_no_event20_no_event67"]["label"].eq(POSITIVE_LABEL), "source_event_id"
    ].astype(int)
)

expected_metric_rows = len(DATASETS) * len(FEATURE_SET_COLUMNS) * 2 * 5
assert len(cv_metrics_df) == expected_metric_rows
assert len(threshold_metrics_df) == len(DATASETS) * len(FEATURE_SET_COLUMNS)
assert len(decision_matrix_df) == len(DATASETS) * len(FEATURE_SET_COLUMNS)
assert set(threshold_metrics_df["threshold"]) == {DECISION_THRESHOLD}

for _, row in cv_metrics_df.iterrows():
    train_groups = set(str(row["train_groups"]).split("|"))
    test_groups = set(str(row["test_groups"]).split("|"))
    assert train_groups.isdisjoint(test_groups)

for path in [PNG_TOP20_MAIN_PATH, PNG_TOP20_NO_EVENT67_PATH, PNG_OVERLAP_PATH]:
    assert path.exists() and path.stat().st_size > 0, path

blocked_terms = ["manufacturer" + "_" + "2", "manufacturer" + " " + "2", "M" + "2"]
for output_path in [
    SUMMARY_PATH,
    CV_METRICS_PATH,
    CV_PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    DECISION_MATRIX_PATH,
    REPORT_PATH,
]:
    text = output_path.read_text(encoding="utf-8")
    assert not any(term in text for term in blocked_terms), output_path

print("validation passed")
for output_path in [
    SUMMARY_PATH,
    CV_METRICS_PATH,
    CV_PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    DECISION_MATRIX_PATH,
    REPORT_PATH,
    PNG_TOP20_MAIN_PATH,
    PNG_TOP20_NO_EVENT67_PATH,
    PNG_OVERLAP_PATH,
]:
    print(output_path.relative_to(ROOT))

validation passed
07_데이터산출물\m1_compact_feature_set_summary.csv
07_데이터산출물\m1_compact_feature_set_cv_metrics.csv
07_데이터산출물\m1_compact_feature_set_cv_predictions.csv
07_데이터산출물\m1_compact_feature_set_threshold_metrics.csv
07_데이터산출물\m1_compact_feature_set_feature_importance.csv
07_데이터산출물\m1_compact_feature_set_decision_matrix.csv
07_데이터산출물\09_M1_pre_event_compact_feature_set_검증_보고서.md
07_데이터산출물\m1_compact_feature_importance_top20_main.png
07_데이터산출물\m1_compact_feature_importance_top20_no_event67.png
07_데이터산출물\m1_compact_feature_set_overlap.png
